In [ ]:
!pip install datasets
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# from datasets import load_dataset
# df = load_dataset("cnn_dailymail","3.0.0")
df = pd.read_csv("/content/Data.csv")

In [ ]:
!pip install keras tensorflow
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [ ]:
class WordAttention(Layer):
    def __init__(self, **kwargs):
        super(WordAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(WordAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)  # Ensure input is float32
        print("Input shape to WordAttention:", x.shape)

        # Get dynamic shape values
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]  # Get num_tokens dynamically from input shape
        feature_dim = x.shape[-1]  # Use the last dimension as feature_dim

        #Calculate uit
        uit = K.tanh(K.dot(x, self.W) + self.b) # (batch_size, num_tokens, feature_dim)
        print("uit shape:", uit.shape)

        #Calculate ait
        ait = K.dot(uit, self.u) # (batch_size, num_tokens, 1)
        print("ait shape:", ait.shape)

        #Apply softmax to get attention weights
        ait = tf.nn.softmax(ait, axis=1) # (batch_size, num_tokens, 1)
        print("ait shape after softmax:", ait.shape)

        #Perform element-wise multiplication
        weighted_input = x * ait # (batch_size, num_tokens, feature_dim)
        print("weighted_input shape:", weighted_input.shape)

        #Sum weighted inputs to get context vector
        output = K.sum(weighted_input, axis=1) # (batch_size, feature_dim)
        print("output shape:", output.shape)

        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
class SentenceAttention(Layer):
    def __init__(self, **kwargs):
        super(SentenceAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(SentenceAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)  # Ensure input is float32
        print("Input shape to SentenceAttention:", x.shape)
        # Ensure correct reshaping based on your specific requirements
        batch_size = tf.shape(x)[0]
        num_sentences = tf.shape(x)[1]
        feature_dim = tf.shape(x)[2] if len(x.shape) > 2 else x.shape[1]

        # Reshaping to match expected input shape, e.g., (batch_size, num_sentences, feature_dim)
        x = tf.reshape(x, (batch_size, num_sentences, feature_dim))

        uit = K.tanh(K.dot(x, self.W) + self.b)
        ait = K.dot(uit, self.u)

        ait = K.squeeze(ait, -1)  # Remove the last axis
        ait = K.expand_dims(ait, -1)  # Add an extra dimension for softmax
        ait = tf.nn.softmax(ait, axis=1)  # Apply softmax along the correct axis

        ait = K.expand_dims(ait, axis=-1)  # Add back a dimension for consistency
        weighted_input = x * ait
        return K.sum(weighted_input, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
# # Build the Hierarchical Attention Model
# def hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
#     # Word-level
#     word_input = Input(shape=(word_count,), dtype='int32')
#     word_embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(word_input)
#     word_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(word_embedding)
#     word_attention = WordAttention()(word_bi_lstm)
#     word_encoder = Model(inputs=word_input, outputs=word_attention)

#     # Sentence-level
#     sentence_input = Input(shape=(sentence_count, word_count), dtype='int32')
#     sentence_encoder = tf.keras.layers.TimeDistributed(word_encoder)(sentence_input)
#     sentence_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(sentence_encoder)
#     sentence_attention = SentenceAttention()(sentence_bi_lstm)

#     return Model(inputs=sentence_input, outputs=sentence_attention)





In [ ]:
train_data, temp_data = train_test_split(df, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

In [ ]:
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_data)

In [ ]:
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [ ]:
# train_data = df["train"]
# val_data = df["validation"]
# test_data = df["test"]

In [ ]:
model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
# def preprocess_data(batch):
#     inputs = tokenizer(
#         batch["article"], max_length=1024, truncation=True, padding="max_length", return_tensors="pt"
#     )
#     labels = tokenizer(
#         batch["highlights"], max_length=128, truncation=True, padding="max_length", return_tensors="pt"
#     )
#     inputs["labels"] = labels["input_ids"]
#     return inputs

In [ ]:
# train_data = train_data.map(preprocess_data, batched=True, remove_columns=["article", "highlights", "id"])
# val_data = val_data.map(preprocess_data, batched=True, remove_columns=["article", "highlights", "id"])

In [ ]:
def hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    word_input = Input(shape=(word_count,), dtype='int32')
    word_embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(word_input)
    word_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(word_embedding)
    word_attention = WordAttention()(word_bi_lstm)
    word_encoder = tf.keras.models.Model(inputs=word_input, outputs=word_attention)

    sentence_input = Input(shape=(sentence_count, word_count), dtype='int32')
    sentence_encoder = tf.keras.layers.TimeDistributed(word_encoder)(sentence_input)
    sentence_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(sentence_encoder)
    sentence_attention = SentenceAttention()(sentence_bi_lstm)

    return tf.keras.models.Model(inputs=sentence_input, outputs=sentence_attention)

In [ ]:
import numpy as np
vocab_size = 10000
embedding_dim = 300
sentence_count = 10
word_count = 20
embedding_matrix = np.random.rand(vocab_size, embedding_dim)

hierarchical_model = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)

In [ ]:
# def preprocess_example(example):
#     text = example['Content']
#     sentences = text.split('.')
#     tokenized_sentences = [tokenizer(sentence)['input_ids'][:word_count] for sentence in sentences]
#     padded_sentences = tf.keras.preprocessing.sequence.pad_sequences(
#         tokenized_sentences, maxlen=word_count, padding='post', truncating='post'
#     )
#     document = tf.keras.preprocessing.sequence.pad_sequences(
#         [padded_sentences], maxlen=sentence_count, padding='post', truncating='post'
#     )
#     return document

In [ ]:
# processed_datasets = {}
# for split in dataset_dict.keys():
#     processed_data = []
#     for example in dataset_dict[split]:
#         input_data = preprocess_example(example)
#         input_data = np.array(input_data)
#         output = hierarchical_model(input_data)
#         processed_data.append(output.numpy())
#     processed_datasets[split] = processed_data

In [ ]:
def preprocess_function(batch, tokenizer, model, vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    batch_size = 4  # Small batch size for Kaggle notebook

    combined_text = [
        f"{headline} {content} {category}"
        for headline, content, category in zip(batch['Headline'], batch['Content'], batch['Category'])
    ]

    # Clear CUDA cache before tokenization
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Encode input text
    inputs = tokenizer(
        combined_text,
        truncation=True,
        padding="max_length",
        max_length=min(512, sentence_count * word_count),
        return_tensors="pt"
    )

    # Encode target summaries
    targets = tokenizer(
        batch['Human Summary'],
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Return only the necessary fields for training
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"]
    }

# Process the dataset
processed_dataset = dataset_dict.map(
    lambda batch: preprocess_function(
        batch, tokenizer, model, vocab_size,
        embedding_dim, sentence_count, word_count,
        embedding_matrix
    ),
    batched=True,
    batch_size=4,
    num_proc=1,
    load_from_cache_file=False
)

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [ ]:
# import numpy as np
# embedding_matrix = np.random.rand(vocab_size, embedding_dim)
# train_dataset = dataset_dict["train"].map(
#     lambda batch: preprocess_function(batch, tokenizer, model, vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix), # Pass to preprocess_function
#     batched=True # Process in batches to improve performance
# )
# validation_dataset = dataset_dict["validation"].map(
#     lambda batch: preprocess_function(batch, tokenizer, model, vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix),
#     batched=True
# )
# test_dataset = dataset_dict["test"].map(
#     lambda batch: preprocess_function(batch, tokenizer, model, vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix),
#     batched=True
# )

In [ ]:
processed_dataset = processed_dataset.remove_columns(dataset_dict["train"].column_names)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-finetuned",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.001,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    predict_with_generate=True,
    remove_unused_columns=False,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

<ipython-input-27-4a1bfd62e1e0>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,9.029300,3.981659
2,8.766200,3.884376
3,9.651900,3.812200
4,9.127200,3.755913
5,8.505500,3.692006
6,7.459100,3.623718
7,7.925000,3.536030
8,6.251300,3.469015
9,6.562800,3.419169


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=440, training_loss=8.044956814159047, metrics={'train_runtime': 1026.1645, 'train_samples_per_second': 0.867, 'train_steps_per_second': 0.429, 'total_flos': 492676256563200.0, 'train_loss': 8.044956814159047, 'epoch': 9.786516853932584})

In [ ]:
def evaluate_summaries(dataset, model, tokenizer):
   references = []
   predictions = []

   for sample in dataset:
       # Combine input attributes similar to preprocessing
       input_text = f"{sample['Headline']} {sample['Content']} {sample['Category']}"
       reference_summary = sample['Human Summary']
       references.append(reference_summary)

       # Generate summary
       inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(model.device)
       summary_ids = model.generate(
           inputs["input_ids"],
           max_length=128,
           num_beams=4,
           early_stopping=True
       )
       generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
       predictions.append(generated_summary)

   return references, predictions

In [ ]:
test_references, test_predictions = evaluate_summaries(test_dataset, model, tokenizer)


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

def calculate_average_precision(test_references, test_predictions):
    # Create a set of all unique words across human and predicted summaries
    all_words = set(word for summary in test_references for word in summary.lower().split()) | \
                set(word for summary in test_predictions for word in summary.lower().split())

    # Convert human summaries and predicted summaries into binary vectors
    human_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_references
    ]
    predicted_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_predictions
    ]

    # Calculate the average precision score
    ap_scores = []
    for true, pred in zip(human_vectors, predicted_vectors):
        try:
            ap_scores.append(average_precision_score(true, pred))
        except ValueError:
            ap_scores.append(0.0)

    return np.mean(ap_scores)

In [ ]:
ap_score = calculate_average_precision(test_references, test_predictions)
print(f"Average Precision Score: {ap_score}")

Average Precision Score: 0.20959777962795975


In [ ]:
!pip install evaluate
!pip install rouge_score
import evaluate
rouge = evaluate.load("rouge")

In [ ]:

results = rouge.compute(predictions=test_predictions, references=test_references)

In [ ]:
print(f"ROUGE-1: {results['rouge1']}")
print(f"ROUGE-2: {results['rouge2']}")
print(f"ROUGE-L: {results['rougeL']}")

ROUGE-1: 0.4562573449818902
ROUGE-2: 0.21693416602972987
ROUGE-L: 0.3220899905208481
